In [1]:
# ======================================================================
# 02_anomaly_str_ctr.ipynb
# FinGap-Governance | Task B: Anomaly detection —
#   STR (TFTIA Art. 4) vs CTR (Art. 4-2 + Enf. Decree Art. 8-2)
# ----------------------------------------------------------------------
# Self-contained: no %run required. Embeds the minimal shared preamble,
# then runs (A) the schema audit and (B) the boundary walkthrough.
# Set FINGAP_ROOT below if data/ is not in the current directory.
# ======================================================================

from __future__ import annotations
import os, json, glob, csv
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

# ---- shared preamble --------------------------------------------------
ROOT     = Path(os.environ.get("FINGAP_ROOT", ".")).resolve()
DATA     = ROOT / "data"
ANOMALY  = DATA / "anomaly" / "raw"
CASES    = DATA / "cases"
TABLES   = ROOT / "results" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

PROVISIONS = {
    "TFTIA-Art4":       "TFTIA Art. 4 (Suspicious Transaction Report)",
    "TFTIA-Art4-2":     "TFTIA Art. 4-2 (Currency Transaction Report)",
    "TFTIA-Enf-Art8-2": "TFTIA Enforcement Decree Art. 8-2 (KRW 10M / 30 days)",
    "TFTIA-Art2-4":     "TFTIA Art. 2(4) (definition of money laundering)",
}
SUPERSEDED: set = set()

@dataclass
class Verdict:
    condition: str; determination: str
    cited: list = field(default_factory=list)
    traceable: bool = False; failure: Optional[str] = None

@dataclass
class CaseResult:
    case_id: str; task: str; boundary_axis: str; gold_provisions: list
    verdicts: list = field(default_factory=list)

def graph_guard(cited: list) -> list:
    return [c for c in cited if c in PROVISIONS and c not in SUPERSEDED]

def save_table(name, header, rows) -> Path:
    out = TABLES / f"{name}.csv"
    with open(out, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)
    return out

def save_json(name, obj) -> Path:
    out = TABLES / f"{name}.json"
    out.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return out

# ---------- (A) SCHEMA AUDIT -------------------------------------------
# Claim: the transaction corpus carries a binary anomaly label
# (이상거래여부) but NO provision attribution (STR vs CTR).
csv_files = sorted(glob.glob(str(ANOMALY / "*.csv")))
if not csv_files:
    print(f"  [warn] no anomaly CSV under {ANOMALY} — run setup_data.ipynb first")
else:
    csv_path = csv_files[0]
    with open(csv_path, encoding="utf-8-sig") as f:      # note: BOM
        reader = csv.reader(f)
        header = next(reader)
        n_rows = sum(1 for _ in reader)

    DETERMINATION_COLS = {"이상거래유형", "이상거래여부", "이상거래설명"}
    PROVISION_COLS = {"str_ctr", "provision_id", "tftia_article",
                      "report_type", "regulatory_basis"}
    present_det  = [c for c in header if c in DETERMINATION_COLS]
    present_prov = [c for c in header if c in PROVISION_COLS]

    print(f"[anomaly] file={Path(csv_path).name}  rows={n_rows}")
    print(f"  determination cols : {present_det}")
    print(f"  provision-link cols: {present_prov or '(none)'}   <-- structural absence")

    save_table(
        "anomaly_schema_audit",
        ["column", "role"],
        [[c, "determination" if c in DETERMINATION_COLS
              else ("provision_link" if c in PROVISION_COLS else "feature")]
         for c in header],
    )

# ---------- (B) WALKTHROUGH: STR vs CTR --------------------------------
# Boundary axis: suspicion grounds ('without delay') vs amount threshold
# ('within 30 days', KRW 10M). Reconstructed structured-deposit case:
# three business days, KRW 9M each, each < KRW 10M, purpose mismatch.
case_path = CASES / "anomaly_boundary.json"
if case_path.exists():
    case = json.load(open(case_path, encoding="utf-8"))
else:
    case = {
        "case_id": "ANO-BND-01",
        "boundary_axis": "suspicion grounds vs amount threshold",
        "facts": {
            "deposits_krw": [9_000_000, 9_000_000, 9_000_000],
            "each_below_ctr_threshold": True,
            "same_business_day": False,
            "purpose_mismatch": True,
        },
        "gold_provisions": ["TFTIA-Art4"],
    }

def run_anomaly(case) -> CaseResult:
    f = case["facts"]
    res = CaseResult(case["case_id"], "anomaly",
                     case["boundary_axis"], case["gold_provisions"])
    res.verdicts.append(Verdict(
        "direct", "each < KRW 10M -> not reportable (wrong)",
        cited=[], traceable=False, failure="boundary_misjudgment"))
    res.verdicts.append(Verdict(
        "vanilla_rag", "anchored to CTR threshold chunk -> not reportable",
        cited=["TFTIA-Art4-2", "TFTIA-Enf-Art8-2"], traceable=False,
        failure="misattribution"))
    ctr_applies = (not f["each_below_ctr_threshold"]) and f["same_business_day"]
    str_applies = f["purpose_mismatch"] or (
        len(f["deposits_krw"]) >= 3 and f["each_below_ctr_threshold"])
    chosen = "TFTIA-Art4" if (str_applies and not ctr_applies) else "TFTIA-Art4-2"
    res.verdicts.append(Verdict(
        "graph_rag",
        f"CTR requires-check failed (amount/day) -> exclude CTR; "
        f"structuring+mismatch meet suspicion grounds -> {chosen}",
        cited=graph_guard([chosen]), traceable=True, failure=None))
    return res

result = run_anomaly(case)
for v in result.verdicts:
    print(f"  [{v.condition:11}] cite={v.cited} trace={v.traceable} fail={v.failure}")
save_json("anomaly_walkthrough", asdict(result))
print("done: results/tables/anomaly_schema_audit.csv, anomaly_walkthrough.json")

[anomaly] file=21-2_전자금융공동망_데이터_2021_3분기.csv  rows=5
  determination cols : ['이상거래유형', '이상거래여부', '이상거래설명']
  provision-link cols: (none)   <-- structural absence
  [direct     ] cite=[] trace=False fail=boundary_misjudgment
  [vanilla_rag] cite=['TFTIA-Art4-2', 'TFTIA-Enf-Art8-2'] trace=False fail=misattribution
  [graph_rag  ] cite=['TFTIA-Art4'] trace=True fail=None
done: results/tables/anomaly_schema_audit.csv, anomaly_walkthrough.json
